# 3. Model sensitivity analysis (CH4 util-free)

**Purpose:** CTS-2026-0235R2 Associate Editor supplement — refit the CH4 (`non_opioid_ed`) model **without utilization-derived features** and compare holdout AUPRC, top drug SHAP ranks, and published pair/triplet persistence vs the primary framing.

**Template:** Same EC2 root-notebook style as [3_model_train_shap_ffa.ipynb](3_model_train_shap_ffa.ipynb) (setup → sync → scoped run → artifacts on disk/S3). This notebook does **not** retrain the full Step 6/7/8 production stack.

**Production runner:** `6_final_model/run_sensitivity_util_free.py` (SSOT).

**Default scope:** `non_opioid_ed` / **all modeled age bands** (same `AGE_BANDS` as Step 6). AE asked for a single supplemental analysis; multi-band coverage is for modeling consistency.

**Outputs (not embedded):**
- `6_final_model/outputs/non_opioid_ed/{age}_util_free/` per age band
- `8_ffa_analysis/outputs/non_opioid_ed/{age}_util_free_sensitivity/` per age band
- `manuscript/data/supplementary/ch04_util_free_sensitivity/` (`sensitivity_summary_all_bands.json`, per-band subfolders)

**Docs:** `manuscript/data/supplementary/ch04_util_free_sensitivity/README.md`

Run from **repo root** on EC2 (NVMe data root via `get_data_root()`).


In [ ]:
# Setup: paths and project root (match notebook 3 template)
import os
import sys
import subprocess
import shutil
from pathlib import Path

PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == "10_risk_dashboard":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "10_risk_dashboard").exists():
    pass
else:
    for cand in (Path.cwd(), *Path.cwd().parents):
        if (cand / "6_final_model").exists() and (cand / "py_helpers").exists():
            PROJECT_ROOT = cand
            break

sys.path.insert(0, str(PROJECT_ROOT))
from py_helpers.env_utils import get_data_root, get_model_data_root

S3_BUCKET = os.environ.get("PGX_S3_BUCKET", "pgxdatalake")
DATA_ROOT = get_data_root()
MODEL_DATA_ROOT = get_model_data_root()
AWS_PROFILE = os.environ.get("AWS_PROFILE")
FINAL_MODEL_OUTPUTS = PROJECT_ROOT / "6_final_model" / "outputs"
SENSITIVITY_SCRIPT = PROJECT_ROOT / "6_final_model" / "run_sensitivity_util_free.py"

print("PGx model sensitivity workflow")
print("=" * 60)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print(f"Model data root: {MODEL_DATA_ROOT}")
print(f"Final model outputs: {FINAL_MODEL_OUTPUTS}")
print(f"Sensitivity script: {SENSITIVITY_SCRIPT}")
print("=" * 60)
assert SENSITIVITY_SCRIPT.exists(), f"Missing runner: {SENSITIVITY_SCRIPT}"


## Config

Default = all Step-6 age bands for `non_opioid_ed`. Edit `AGE_BANDS_RUN` only for debugging subsets.


In [ ]:
# Sensitivity run scope — all modeled age bands (consistent with Step 6)
from py_helpers.constants import AGE_BANDS, age_band_to_fname

COHORT = "non_opioid_ed"
AGE_BANDS_RUN = list(AGE_BANDS)
# Debug subset example: AGE_BANDS_RUN = ["65-74", "75-84", "85-114"]

SYNC_FINAL_MODEL_FROM_S3 = True
SYNC_MODEL_EVENTS_FROM_S3 = False

MS_DIR = PROJECT_ROOT / "manuscript" / "data" / "supplementary" / "ch04_util_free_sensitivity"

print(f"Cohort: {COHORT}")
print(f"Age bands ({len(AGE_BANDS_RUN)}): {AGE_BANDS_RUN}")
print(f"Manuscript supplementary: {MS_DIR}")


## Sync required inputs from S3 (idempotent)

Sync **Step 6 final-model** gold artifacts for the configured cohort/age band so the util-free refit can use the same holdout feature tables / metrics when present. Uses `aws s3 sync` (profile from `AWS_PROFILE` when set).


In [ ]:
# Sync final_model gold prefixes for each age band (idempotent)
_aws = shutil.which("aws") or "aws"
_profile = ["--profile", AWS_PROFILE] if AWS_PROFILE else []

if SYNC_FINAL_MODEL_FROM_S3:
    for age_band in AGE_BANDS_RUN:
        age_fname = age_band_to_fname(age_band)
        local_dest = FINAL_MODEL_OUTPUTS / COHORT / age_fname
        local_dest.mkdir(parents=True, exist_ok=True)
        for prefix in (
            f"gold/final_model/{COHORT}/{age_band}",
            f"gold/manuscript/final_model/{COHORT}/{age_band}",
        ):
            uri = f"s3://{S3_BUCKET}/{prefix}"
            print(f"Sync {uri} -> {local_dest}")
            r = subprocess.run(
                [_aws, "s3", "sync", uri, str(local_dest)] + _profile,
                capture_output=True,
                text=True,
            )
            if r.returncode == 0:
                print(f"  OK ({prefix})")
            else:
                print(f"  skip/warn exit={r.returncode}: {(r.stderr or r.stdout or '')[:300]}")
else:
    print("SYNC_FINAL_MODEL_FROM_S3=False; using local artifacts only")

if SYNC_MODEL_EVENTS_FROM_S3:
    dest = Path(MODEL_DATA_ROOT)
    dest.mkdir(parents=True, exist_ok=True)
    for age_band in AGE_BANDS_RUN:
        me_prefix = f"gold/cohorts_model_data/cohort_name={COHORT}/age_band={age_band}"
        uri = f"s3://{S3_BUCKET}/{me_prefix}"
        print(f"Sync {uri} -> {dest}")
        r = subprocess.run(
            [_aws, "s3", "sync", uri, str(dest)] + _profile,
            capture_output=True,
            text=True,
        )
        print("  OK" if r.returncode == 0 else f"  warn exit={r.returncode}")


## Run utilization-free sensitivity (CH4, all age bands)

Calls `6_final_model/run_sensitivity_util_free.py` → `main(AGE_BANDS_RUN)`. Writes per-band artifacts + rollup JSON/CSV under manuscript supplementary.


In [ ]:
# Run production sensitivity runner for all configured age bands
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "6_final_model"))
import run_sensitivity_util_free as sens  # noqa: E402

print(f"Executing util-free sensitivity for {len(AGE_BANDS_RUN)} age bands ...")
ROLLUP = sens.main(AGE_BANDS_RUN)
print("Sensitivity runner finished.")
print(f"Succeeded: {ROLLUP.get('n_succeeded')}/{ROLLUP.get('n_requested')}")
if ROLLUP.get("age_bands_failed"):
    print("Failures:", ROLLUP["age_bands_failed"])


## Load rollup summary

Prefer `sensitivity_summary_all_bands.json` / `sensitivity_auprc_by_age_band.csv`. Do not paste full SHAP tables into the notebook.


In [ ]:
# Print headline rollup only (no large embeds)
import json

json_path = MS_DIR / "sensitivity_summary_all_bands.json"
csv_path = MS_DIR / "sensitivity_auprc_by_age_band.csv"

assert json_path.exists(), f"Missing rollup: {json_path}"
rollup = json.loads(json_path.read_text(encoding="utf-8"))
print("Rollup:", json_path)
print(f"  requested: {rollup.get('n_requested')}  succeeded: {rollup.get('n_succeeded')}")
print(f"  failed: {list((rollup.get('age_bands_failed') or {}).keys())}")
print("\nPer-band headlines:")
for band, row in sorted((rollup.get("per_band") or {}).items()):
    print(
        f"  {band}: auprc={row.get('util_free_auprc')} "
        f"lift={row.get('util_free_pr_lift')} "
        f"jaccard={row.get('drug_shap_jaccard')} "
        f"pairs={row.get('published_pairs_positive_ie')}"
    )
if csv_path.exists():
    print("\nTable:", csv_path)
print("Artifacts root:", MS_DIR)
